<a href="https://colab.research.google.com/github/Muskan624-ai/bias-detector/blob/main/notebooks/Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re
import torch
import shutil
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

Mounted at /content/drive


In [2]:
print("⏳ Loading master dataset...")
df = pd.read_csv('/content/drive/MyDrive/BiasProject/data.csv')

# Drop any accidental blank rows
df = df.dropna(subset=['text', 'label'])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

print("🧹 Sanitizing text data...")
df['text'] = df['text'].apply(clean_text)

# Stratified Split to maintain exact balanced target ratios
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"\n✅ Data Prepared!")
print(f"Total rows in dataset: {len(df)}")
print(f"Training on: {len(train_texts)} sentences")
print(f"Testing on: {len(test_texts)} sentences")

⏳ Loading master dataset...
🧹 Sanitizing text data...

✅ Data Prepared!
Total rows in dataset: 12836
Training on: 10268 sentences
Testing on: 2568 sentences


In [3]:
print("\n⏳ Initializing Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased", use_fast=True)

print("Tokenizing training array...")
train_encodings = tokenizer(list(train_texts), truncation=True, max_length=128)
print("Tokenizing testing array...")
test_encodings = tokenizer(list(test_texts), truncation=True, max_length=128)

# Initialize Data Collator for dynamic padding per batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ Tokenization Configuration Complete!")


⏳ Initializing Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing training array...
Tokenizing testing array...
✅ Tokenization Configuration Complete!


In [4]:
class BiasDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BiasDataset(train_encodings, train_labels)
test_dataset = BiasDataset(test_encodings, test_labels)
print("✅ PyTorch Datasets ready for model input!")

✅ PyTorch Datasets ready for model input!


In [5]:
print("\n⏳ Loading Pre-trained DistilBERT Architecture (4 Target Classes)...")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

# Verify if CUDA / T4 GPU is running
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✅ Model moved to operational device: {device}")
if device.type != 'cuda':
    print("⚠️ WARNING: Colab is running on CPU. Please go to Runtime -> Change runtime type -> Select T4 GPU!")

print("⏳ Engineering dynamic class weights matrix...")
# Calculate exact inverse weights based on your training labels distribution
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels.values
)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print(f"🎯 Calculated Class Weights Matrix (0, 1, 2, 3): {weights}")


⏳ Loading Pre-trained DistilBERT Architecture (4 Target Classes)...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model moved to operational device: cuda
⏳ Engineering dynamic class weights matrix...
🎯 Calculated Class Weights Matrix (0, 1, 2, 3): [0.33476787 1.61345066 3.9675425  7.09116022]


In [6]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        # Forward pass through DistilBERT
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Deploy custom weighted loss function using engineered weights
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [7]:
from transformers import EarlyStoppingCallback

# 1. Update TrainingArguments to evaluate every step instead of every epoch
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.1,
    eval_strategy="steps",           # <--- CHANGE TO STEPS
    eval_steps=100,                  # <--- EVALUATE EVERY 100 STEPS
    save_strategy="steps",           # <--- MUST MATCH EVAL STRATEGY
    save_steps=100,
    logging_steps=50,
    report_to="none",
    fp16=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True      # <--- REQUIRED FOR EARLY STOPPING
)

# 2. Add the callback to the Trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # <--- ADD THIS
)
trainer.train()
print("🎉 Class-Weighted Model Training Cycle Complete!")

Step,Training Loss,Validation Loss
100,1.371935,1.354124
200,1.216938,1.085576
300,0.683424,0.551538
400,0.362669,0.338637
500,0.279972,0.270908
600,0.240785,0.246601
700,0.203913,0.267901
800,0.165360,0.240338
900,0.174795,0.279011
1000,0.122717,0.281195


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Class-Weighted Model Training Cycle Complete!


In [8]:
import numpy as np
import torch
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. Ensure model is in evaluation mode
model.eval()

print("⏳ Running predictions on the test dataset...")
# 2. Extract predictions using the trainer
predictions, labels, _ = trainer.predict(test_dataset)

# 3. Convert raw logit predictions to class labels (0, 1, 2, 3)
preds = np.argmax(predictions, axis=1)

# 4. Calculate Mathematical Metrics
accuracy = accuracy_score(labels, preds)
f1_weighted = f1_score(labels, preds, average='weighted')
f1_macro = f1_score(labels, preds, average='macro')

print("\n" + "="*50)
print(f"🎯 FINAL MODEL ACCURACY: {accuracy:.4f}")
print(f"📊 F1 SCORE (Weighted):  {f1_weighted:.4f}")
print(f"📊 F1 SCORE (Macro):     {f1_macro:.4f}")
print("="*50 + "\n")

# 5. Print a deep-dive classification report per class
print("Detailed Class Breakdown:")
print(classification_report(labels, preds, target_names=["Neutral (0)", "Political (1)", "Gender (2)", "Racial (3)"]))

⏳ Running predictions on the test dataset...



🎯 FINAL MODEL ACCURACY: 0.8902
📊 F1 SCORE (Weighted):  0.8955
📊 F1 SCORE (Macro):     0.8516

Detailed Class Breakdown:
               precision    recall  f1-score   support

  Neutral (0)       0.98      0.87      0.92      1918
Political (1)       0.67      0.94      0.78       398
   Gender (2)       0.84      0.93      0.88       162
   Racial (3)       0.70      0.99      0.82        90

     accuracy                           0.89      2568
    macro avg       0.80      0.93      0.85      2568
 weighted avg       0.91      0.89      0.90      2568



In [9]:
print("\n⏳ Archiving weights locally inside Colab workspace...")
model.save_pretrained("./bias_model_final")
tokenizer.save_pretrained("./bias_model_final")

shutil.make_archive("bias_model_final", 'zip', "./bias_model_final")
print("✅ Local zip file generated.")

print("⏳ Exporting final model weights zip to Google Drive...")
shutil.copy("bias_model_final.zip", "/content/drive/MyDrive/BiasProject/bias_model_final.zip")
print("🎯 Done! The highly accurate, class-balanced model weights are securely stored on your Google Drive.")


⏳ Archiving weights locally inside Colab workspace...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Local zip file generated.
⏳ Exporting final model weights zip to Google Drive...
🎯 Done! The highly accurate, class-balanced model weights are securely stored on your Google Drive.
